In [ ]:
!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv
!rm ~/.cache/matplotlib -rf

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
fonts-nanum is already the newest version (20200506-1).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.
/usr/share/fonts: caching, new cache contents: 0 fonts, 1 dirs
/usr/share/fonts/truetype: caching, new cache contents: 0 fonts, 3 dirs
/usr/share/fonts/truetype/humor-sans: caching, new cache contents: 1 fonts, 0 dirs
/usr/share/fonts/truetype/liberation: caching, new cache contents: 16 fonts, 0 dirs
/usr/share/fonts/truetype/nanum: caching, new cache contents: 12 fonts, 0 dirs
/usr/local/share/fonts: caching, new cache contents: 0 fonts, 0 dirs
/root/.local/share/fonts: skipping, no such directory
/root/.fonts: skipping, no such directory
/usr/share/fonts/truetype: skipping, looped directory detected
/usr/share/fonts/truetype/humor-sans: skipping, looped directory detected
/usr/share/fonts/truetype/liberation: skipping, looped directory detected
/usr/share/fonts/truetype/n

In [ ]:
import matplotlib
print(matplotlib.get_cachedir())

/root/.cache/matplotlib


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.rc('font', family='NanumGothic')
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# 1. 설치된 나눔고딕 폰트의 파일 경로
font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'

# 2. Matplotlib의 폰트 매니저에 폰트 강제 추가 (핵심!)
fm.fontManager.addfont(font_path)

# 3. 폰트 이름 추출 및 적용
font_name = fm.FontProperties(fname=font_path).get_name()
plt.rc('font', family=font_name)
plt.rcParams['axes.unicode_minus'] = False  # 마이너스 기호 깨짐 방지

print("✅ 폰트 적용 완료:", font_name)

✅ 폰트 적용 완료: NanumGothic


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


CSV 불러오기

In [ ]:
import pandas as pd

orders_df = pd.read_csv('/content/drive/MyDrive/Sesac 데이터분석가 양성과정/프로젝트_team_level_up/EDA/DATA/orders.csv')
order_payments_df = pd.read_csv('/content/drive/MyDrive/Sesac 데이터분석가 양성과정/프로젝트_team_level_up/EDA/DATA/order_payments.csv')
customer_df = pd.read_csv('/content/drive/MyDrive/Sesac 데이터분석가 양성과정/프로젝트_team_level_up/EDA/DATA/customers.csv')
order_items_df = pd.read_csv('/content/drive/MyDrive/Sesac 데이터분석가 양성과정/프로젝트_team_level_up/EDA/DATA/order_items.csv')
products_df = pd.read_csv('/content/drive/MyDrive/Sesac 데이터분석가 양성과정/프로젝트_team_level_up/EDA/DATA/products.csv')
category_name_df = pd.read_csv('/content/drive/MyDrive/Sesac 데이터분석가 양성과정/프로젝트_team_level_up/EDA/DATA/product_category_name_translation.csv')
order_reviews_df = pd.read_csv('/content/drive/MyDrive/Sesac 데이터분석가 양성과정/프로젝트_team_level_up/EDA/DATA/order_reviews.csv')
sellers_df = pd.read_csv('/content/drive/MyDrive/Sesac 데이터분석가 양성과정/프로젝트_team_level_up/EDA/DATA/sellers.csv')
geolocation_df = pd.read_csv('/content/drive/MyDrive/Sesac 데이터분석가 양성과정/프로젝트_team_level_up/EDA/DATA/geolocation.csv')

테이블 합치기

In [ ]:
olist_df = pd.merge(orders_df, order_payments_df, on = 'order_id')
olist_df = olist_df.merge(customer_df, on = 'customer_id')
olist_df = olist_df.merge(order_items_df, on = 'order_id')
olist_df = olist_df.merge(products_df, on = 'product_id')
olist_df = olist_df.merge(category_name_df, on = 'product_category_name')
olist_df = olist_df.merge(order_reviews_df, on = 'order_id')
olist_df = olist_df.merge(sellers_df, on = 'seller_id')

# 컬럼 이름 변경
olist_df = olist_df.rename(columns={'geolocation_state': 'seller_state'})

# 변경 확인
print(olist_df.info())

olist_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 114812 entries, 0 to 114811
Data columns (total 39 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   order_id                       114812 non-null  object 
 1   customer_id                    114812 non-null  object 
 2   order_status                   114812 non-null  object 
 3   order_purchase_timestamp       114812 non-null  object 
 4   order_approved_at              114798 non-null  object 
 5   order_delivered_carrier_date   114087 non-null  object 
 6   order_delivered_customer_date  112943 non-null  object 
 7   order_estimated_delivery_date  114812 non-null  object 
 8   payment_sequential             114812 non-null  int64  
 9   payment_type                   114812 non-null  object 
 10  payment_installments           114812 non-null  int64  
 11  payment_value                  114812 non-null  float64
 12  customer_unique_id            

> [전처리 1] 날짜 데이터(String)를 실제 시간 데이터(Datetime)로 변환

In [ ]:
import pandas as pd
from datetime import timedelta

# 분석에 사용할 주요 날짜 컬럼 목록
DATE_COLS = [
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

# 1. 텍스트를 날짜(datetime) 타입으로 일괄 변환 (에러 발생 시 강제로 결측치 처리)
for c in DATE_COLS:
    olist_df[c] = pd.to_datetime(olist_df[c], errors='coerce')

# 2. 날짜 연산 및 중복 결제 병합을 위해 시간(시간:분:초)을 떼어낸 순수 날짜(Date) 컬럼 생성
olist_df['purchase_date'] = olist_df['order_purchase_timestamp'].dt.date

print("✅ [전처리 1 완료] 날짜 데이터 변환 완료")

✅ [전처리 1 완료] 날짜 데이터 변환 완료


> [전처리 2] 타겟 생성: 분리결제 노이즈 제거 및 30일 이내 진성 재구매(target_30d) 식별

In [ ]:
# 1. 고객별 첫 구매일(min)과, 고유 구매일자 리스트(order_dates) 생성
customer_orders = olist_df.groupby('customer_unique_id').agg(
    first_order_date=('purchase_date', 'min'),
    # 같은 날짜 중복 제거를 위해 set() 사용 후 정렬
    order_dates=('purchase_date', lambda x: sorted(list(set(x))))
).reset_index()

# 2. 30일 이내 재구매 여부 판별 함수 정의
def check_30d_repurchase(dates):
    if len(dates) < 2:  # 구매일이 하루밖에 없는 사람 (단발성)
        return 0
    # 첫 구매일과 두 번째 구매일의 차이가 0초과 30일 이내면 1(재구매)
    return 1 if 0 < (dates[1] - dates[0]).days <= 30 else 0

# 3. 함수 적용하여 타겟 변수 생성
customer_orders['target_30d'] = customer_orders['order_dates'].apply(check_30d_repurchase)

print("✅ [전처리 2 완료] 진성 재구매 타겟 변수 생성 완료")

✅ [전처리 2 완료] 진성 재구매 타겟 변수 생성 완료


> [전처리 3] 관측창 통제: 분석 기간 종료 직전 가입자 제외 및 최종 데이터셋 완성

In [ ]:
# 1. 전체 데이터셋의 가장 마지막 주문일 확인
max_dataset_date = olist_df['purchase_date'].max()

# 2. 마지막 주문일 기준, 최소 30일 이전에 첫 구매를 한 '시간적 여유가 있는' 고객만 필터링
# (이전에 에러가 났던 부분을 방지하기 위해 파이썬 내장 timedelta 사용)
valid_customers = customer_orders[customer_orders['first_order_date'] <= (max_dataset_date - timedelta(days=30))]

# 3. 만약 기존 olist_df에 target_30d가 잘못 붙어있다면 꼬임 방지를 위해 삭제
if 'target_30d' in olist_df.columns:
    olist_df = olist_df.drop(columns=['target_30d'])

# 4. 필터링된 유효 고객(valid_customers)과 그들의 타겟 변수를 원본 olist_df에 이너 조인(Inner Join)
olist_df = olist_df.merge(valid_customers[['customer_unique_id', 'target_30d']], on='customer_unique_id', how='inner')

print("✅ [전처리 3 완료] 관측창 통제 및 타겟 결합 완료")
print("   - 관측창 확보된 유효 데이터 크기:", olist_df.shape)

✅ [전처리 3 완료] 관측창 통제 및 타겟 결합 완료
   - 관측창 확보된 유효 데이터 크기: (108941, 41)


> [전처리 4] 가설 검증용 파생변수 생성 (배송 지연, 배송비 비율 등)

In [ ]:
# 1. [통제변수] 배송 지연 여부 (실제 배송일 > 배송 예정일 이면 1, 아니면 0)
olist_df['is_late'] = (olist_df['order_delivered_customer_date'] > olist_df['order_estimated_delivery_date']).astype(int)

# 2. [가설 4] 배송비 부담률 = 배송비 / (상품가격 + 배송비)
# 분모가 0이 되어 무한대(Inf)가 나오는 것을 방지하기 위해 아주 작은 수(1e-5)를 더해줌
olist_df['freight_ratio'] = olist_df['freight_value'] / (olist_df['price'] + olist_df['freight_value'] + 1e-5)

print("✅ [전처리 4 완료] 가설 검증용 파생 변수 생성 완료")

✅ [전처리 4 완료] 가설 검증용 파생 변수 생성 완료


> [전처리 5] 중복제거 (데이터 누수 차단 및 EDA 데이터셋 분리)

In [ ]:
# 1. 고객별 가장 처음 발생한 '첫 주문번호(order_id)' 찾기
first_orders = olist_df.sort_values('order_purchase_timestamp').groupby('customer_unique_id')['order_id'].first().reset_index()

# 2. 첫 주문번호에 해당하는 데이터(미래의 두 번째 결제 내역 차단)만 추출
first_purchase_df = olist_df[olist_df['order_id'].isin(first_orders['order_id'])]

# 3. 중복 제거의 올바른 방법: 고객 1명당 1줄의 요약본(eda_df)으로 집계(Aggregation)
# 이 과정에서 '가설 1. 카테고리 다양성' 검증을 위한 nunique()가 계산됩니다.
eda_df = first_purchase_df.groupby('customer_unique_id').agg(
    category_count=('product_category_name', 'nunique'),     # [가설 1] 장바구니 속 카테고리 종류 수
    purchase_date=('order_purchase_timestamp', 'first'),     # [가설 2] 구매 일자
    installments=('payment_installments', 'max'),            # [가설 3] 최대 할부 개월 수
    freight_ratio=('freight_ratio', 'mean'),                 # [가설 4] 장바구니 평균 배송비 부담률
    review_score=('review_score', 'first'),                  # [가설 5] 첫 리뷰 점수
    is_late=('is_late', 'max'),                              # 배송 지연 경험 여부 (하나라도 지연됐으면 1)
    target_30d=('target_30d', 'first')                       # 30일 이내 진성 재구매 여부
).reset_index()

print("✅ [전처리 5 완료] 고객 단위(1인 1행) EDA용 요약 데이터(eda_df) 구축 완료")
print("   - 최종 EDA 데이터셋 크기:", eda_df.shape)

✅ [전처리 5 완료] 고객 단위(1인 1행) EDA용 요약 데이터(eda_df) 구축 완료
   - 최종 EDA 데이터셋 크기: (87748, 8)


In [ ]:
olist_df.reset_index(drop = True, inplace = True)
olist_df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,payment_sequential,payment_type,...,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,seller_zip_code_prefix,seller_state,purchase_date,target_30d,is_late,freight_ratio
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,1,credit_card,...,NaN,"Não testei o produto ainda, mas ele veio corre...",2017-10-11 00:00:00,2017-10-12 03:43:48,9350,SP,2017-10-02,1,0,0.225265
1,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,3,voucher,...,NaN,"Não testei o produto ainda, mas ele veio corre...",2017-10-11 00:00:00,2017-10-12 03:43:48,9350,SP,2017-10-02,1,0,0.225265
2,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,2,voucher,...,NaN,"Não testei o produto ainda, mas ele veio corre...",2017-10-11 00:00:00,2017-10-12 03:43:48,9350,SP,2017-10-02,1,0,0.225265
3,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,1,boleto,...,Muito boa a loja,Muito bom o produto.,2018-08-08 00:00:00,2018-08-08 18:37:50,31570,MG,2018-07-24,0,0,0.160894
4,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,1,credit_card,...,NaN,O produto foi exatamente o que eu esperava e e...,2017-12-03 00:00:00,2017-12-05 19:21:58,31842,MG,2017-11-18,0,0,0.376731
